# Callbacks & Guardrails

Intercept tool calls with a before_tool_callback.

> Part of the **ADK Katas**. Work top-to-bottom: read the concept, fill in the
> `# TODO`s in the starter cell, then run the **Check** cell — it grades your
> code offline (no API call). The **Run it live** cell needs a `GOOGLE_API_KEY`.


## Kata 05 — Callbacks & Guardrails

**Callbacks** let you hook the agent's lifecycle. A `before_tool_callback`
runs *just before* a tool executes — return `None` to allow it, or return a
dict to **short-circuit** the tool with that value (a guardrail).

```python
def block_tool(tool, args, tool_context):
    if args.get("city", "").lower() in BLOCKED:
        return {"status": "error", "error_message": "That city is blocked."}
    return None  # allow
```

## Your task

1. Implement `guardrail(tool, args, tool_context)`: if `args["city"]`
   (case-insensitive) is in `BLOCKED`, return an **error dict**; otherwise
   return `None`.
2. Attach it to `root_agent` via `before_tool_callback=guardrail`.

In [ ]:
# Setup — run me first
import sys, pathlib
# Make the kata library importable whether opened from adk-katas/ or adk-katas/notebooks/
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent):
    if (_p / "kata_helpers.py").exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from kata_helpers import *          # load_env, has_api_key, run_agent, check, grade, requires_key
from kata_content import BY_SLUG

KATA = BY_SLUG["callbacks-guardrails"]
load_env()
print("API key configured:" , has_api_key())


In [ ]:
# ✏️ Your code — fill in the TODOs
from google.adk.agents import Agent
from tools.city_tools import get_weather

BLOCKED = {"mordor", "atlantis"}

def guardrail(tool, args, tool_context):
    # TODO: block weather lookups for any city in BLOCKED by returning
    #       {"status": "error", "error_message": "..."}; otherwise return None.
    return None

# TODO: build root_agent with tools=[get_weather] and
#       before_tool_callback=guardrail.
root_agent = None

In [ ]:
# ✅ Check your work (offline — grades the symbols you defined above)
results = KATA.run_checks(globals())
grade([check(r.label, r.passed, r.detail) for r in results])


In [ ]:
# ▶️ Run it live (needs GOOGLE_API_KEY). Each run is a fresh single-turn session.
agent = globals().get(KATA.chat_symbol)

if not KATA.chattable:
    print("This kata has no chat agent — the Check cell is the goal. 🎯")
elif has_api_key() and agent is not None:
    result = await run_agent(agent, "What's the weather in Mordor?")   # noqa: F704  (top-level await is fine in Jupyter)
    print("Response:", result.text)
    if result.tool_calls:
        print("Tools called:", result.tool_calls, result.tool_args)
    if result.transfers:
        print("Transferred to:", result.transfers)
    if result.state:
        print("Session state:", result.state)
else:
    requires_key(lambda: None)


---
### Stuck? Reveal the reference solution

<details>
<summary>Show solution</summary>

```python
from google.adk.agents import Agent
from tools.city_tools import get_weather

BLOCKED = {"mordor", "atlantis"}

def guardrail(tool, args, tool_context):
    city = (args or {}).get("city", "")
    if city.lower() in BLOCKED:
        return {
            "status": "error",
            "error_message": f"Weather lookups for '{city}' are blocked.",
        }
    return None

root_agent = Agent(
    name="guarded_weather_bot",
    model="gemini-2.5-flash",
    instruction="Answer weather questions using get_weather.",
    tools=[get_weather],
    before_tool_callback=guardrail,
)
```

</details>

When you're done, try the same kata in the React app's live chat (`./dev.sh`
from the repo root) to watch the tool-call traces.
